In [ ]:
# Session parameters: edit once, then Runtime > Run all on every session.
GITHUB_USER = "jyotishmann"
REPO_NAME = "alignforge"
GIT_REF = "main"  # branch, tag, or commit SHA to train from
DRIVE_ROOT = "/content/drive/MyDrive/alignforge"

# Config paths, relative to the repo root.
MODEL_CFG = "configs/model/qwen2_5_1_5b.yaml"
SFT_DATA_CFG = "configs/data/sft_dev_assistant.yaml"
DPO_DATA_CFG = "configs/data/dpo_dev_assistant.yaml"
SFT_CFG = "configs/train/sft_qlora.yaml"
SFT_SMOKE_CFG = "configs/train/sft_smoke.yaml"
DPO_CFG = "configs/train/dpo_qlora.yaml"
EXPORT_CFG = "configs/export/gguf_q4km.yaml"
BASE_WEIGHTS = "Qwen/Qwen2.5-1.5B-Instruct"
JUDGE_MODEL = "llama-3.3-70b-versatile"
JUDGE_BASE_URL = "https://api.groq.com/openai/v1"
JUDGE_MODE = "skip"  # "skip" now | "echo" = offline plumbing check | "real" = needs API key

# Behaviour switches.
REQUIRE_GPU = True  # False only for a CPU session that just judges/reports
IGNORE_AUDIT = False  # True = run despite FIX findings (not recommended)
SKIP_EMBEDDING = False  # True = keyword-only domain filter (faster, less precise)
EXPECTED_DOMAIN_CASES = 50
RUN_SMOKE_TESTS = True
RUN_BETA_SWEEP = False
RUN_HF_UPLOAD = False
HF_REPO_ID = "your-hf-username/alignforge-dpo-1.5b"
PUSH_RESULTS = False  # needs a GH_TOKEN secret with Contents: write
GIT_NAME, GIT_EMAIL = "Jyotishman Sarma", "jyotishmann@gmail.com"

# Re-run a stage even though state.json says it is done.
FORCE = {
    "data": False,
    "sft": False,
    "dpo": False,
    "publish": False,
    "generate": False,
    "judge": False,
    "export": False,
}

In [ ]:
# Fail fast on a CPU runtime: saves a 5-minute install before discovering it.
import shutil
import subprocess

gpu = ""
if shutil.which("nvidia-smi"):  # absent entirely on CPU runtimes
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    gpu = smi.stdout.strip() if smi.returncode == 0 else ""
print(f"GPU: {gpu or 'none'}")
if REQUIRE_GPU and not gpu:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")

In [ ]:
# Mount Google Drive: the only storage that survives a disconnect.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone or fast-forward the repo. Private repo: add a GH_TOKEN secret (key icon, left bar).
import base64
import pathlib

from google.colab import userdata


def secret(name: str) -> str | None:
    """A Colab secret, or None if missing / notebook access not granted."""
    try:
        return userdata.get(name)
    except Exception:
        return None


REPO = pathlib.Path("/content") / REPO_NAME
CLEAN_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
_tok = secret("GH_TOKEN")
# Token rides in a per-command header: never written to .git/config or printed.
GIT_AUTH = (
    [
        "-c",
        "http.https://github.com/.extraheader=AUTHORIZATION: basic "
        + base64.b64encode(f"x-access-token:{_tok}".encode()).decode(),
    ]
    if _tok
    else []
)


def git(*args: str, cwd: pathlib.Path | None = REPO) -> str:
    res = subprocess.run(["git", *GIT_AUTH, *args], cwd=cwd, capture_output=True, text=True)
    if res.returncode:
        err = res.stderr.replace(_tok, "***") if _tok else res.stderr
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    return res.stdout.strip()


if not (REPO / ".git").exists():
    git("clone", "--quiet", CLEAN_URL, str(REPO), cwd=None)
git("fetch", "--quiet", "--tags", "origin")
git("checkout", "--quiet", GIT_REF)
if GIT_REF in git("branch", "--format=%(refname:short)").split():
    git("merge", "--quiet", "--ff-only", f"origin/{GIT_REF}")
GIT_SHA = git("rev-parse", "--short", "HEAD")
%cd {REPO}
print(f"{REPO} @ {GIT_REF} ({GIT_SHA})")

In [ ]:
# Load the stdlib-only session helpers and point AlignForge's paths at Drive.
import importlib
import sys

sys.path.insert(0, str(REPO / "notebooks"))
import _colab as af

importlib.reload(af)  # pick up a fresh pull without restarting the runtime
L = af.Layout(repo=REPO, drive_root=pathlib.Path(DRIVE_ROOT))
L.ensure()
S = af.State(L.state_file)
for k, v in af.export_env(L).items():
    print(f"{k:26s} {v}")

In [ ]:
# Install training extras under the pinned constraint set. Skipped if this VM already has them.
import hashlib

_inputs = ("pyproject.toml", "constraints/colab-gpu.txt")
_stamp_src = b"".join((REPO / p).read_bytes() for p in _inputs)
STAMP = pathlib.Path("/content/.af_installed_" + hashlib.sha256(_stamp_src).hexdigest()[:12])
if STAMP.exists():
    print("Dependencies already installed on this VM.")
else:
    af.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[train,data]",
            "-c",
            "constraints/colab-gpu.txt",
        ],
        stage="install",
        layout=L,
    )
    STAMP.touch()

In [ ]:
# Secrets -> environment inherited by every child process. Values are never printed.
import os

for name in ("HF_TOKEN", "ALIGNFORGE_JUDGE_API_KEY"):
    value = secret(name)
    if value:
        os.environ[name] = value
    print(f"{name:26s} {'set' if value else 'missing'}")

In [ ]:
# Restore durable state from Drive, then show where the project stands.
print("registry :", af.restore_registry(L))
print("mirrored :", af.restore_mirror(L))
S.set(last_git_sha=GIT_SHA, last_gpu=gpu or "none")
af.status(L, S)

In [ ]:
# Pre-GPU gates: static audit, API preflight (in a child), then the CLI's own doctor.
af.run(
    [sys.executable, "notebooks/_colab.py", "audit", str(REPO)],
    stage="audit",
    layout=L,
    check=not IGNORE_AUDIT,
)
af.run(
    [sys.executable, "notebooks/_colab.py", "preflight", *([] if REQUIRE_GPU else ["--cpu-ok"])],
    stage="preflight",
    layout=L,
)
af.run(["alignforge", "doctor"], stage="doctor", layout=L)

In [ ]:
# Stage 0: eval suites exist BEFORE the data build, so decontamination can see them.
evals = REPO / "evals"
domain = evals / "domain_v1.jsonl"
suites = [p.name for p in evals.rglob("*.jsonl") if p != domain]
if suites and not FORCE["data"]:
    print("Eval suites present:", suites)
else:
    af.run([sys.executable, "scripts/build_eval_suites.py"], stage="eval_suites", layout=L)

n_domain = 0
if domain.exists():
    n_domain = sum(1 for line in domain.open(encoding="utf-8") if line.strip())
print(f"domain_v1.jsonl: {n_domain} cases")
if n_domain == 0:
    raise RuntimeError("evals/domain_v1.jsonl is missing or empty (write cases per Part 07).")
if n_domain < EXPECTED_DOMAIN_CASES:
    print(
        f"WARNING: expected {EXPECTED_DOMAIN_CASES}. Cases added AFTER the data build are not "
        "decontaminated against. Finish them now, or rebuild data later with FORCE['data']."
    )
print(af.persist(L))

In [ ]:
# Stage 1: build SFT and preference datasets; record their content hashes in state.json.
def build_dataset(cfg_path: str, key: str) -> None:
    if S.get(key) and not FORCE["data"]:
        print(f"{key} = {S.get(key)} (already built)")
        return
    before = af.list_manifests(L.data_dir)
    cmd = ["alignforge", "data", "build", "--config", cfg_path]
    if SKIP_EMBEDDING:
        cmd.append("--skip-embedding")
    out = af.run(cmd, stage=f"data_{key}", layout=L)
    S.set(**{key: af.detect_new_dataset_hash(before, L.data_dir, out)})
    print(f"{key} = {S.get(key)}")


build_dataset(SFT_DATA_CFG, "sft_hash")
build_dataset(DPO_DATA_CFG, "pref_hash")
print(af.persist(L))

In [ ]:
# Stage 2a: SFT dry run. Check the table: T4, fp16, epochs=3, lr=2e-4, effective batch 16.
SFT_CMD = [
    "alignforge",
    "train",
    "sft",
    "--config",
    SFT_CFG,
    "--model-config",
    MODEL_CFG,
    "--dataset-hash",
    S.require("sft_hash"),
]
af.run([*SFT_CMD, "--dry-run"], stage="sft_dry", layout=L)

In [ ]:
# Stage 2b: SFT smoke test. 20 rows through the REAL model, callbacks and save path.
# Hermetic: throwaway artifacts dir + registry copy, so nothing durable is touched.
if RUN_SMOKE_TESTS and not S.get("sft_smoke_ok"):
    af.run(
        [
            "alignforge",
            "train",
            "sft",
            "--config",
            SFT_SMOKE_CFG,
            "--model-config",
            MODEL_CFG,
            "--dataset-hash",
            S.require("sft_hash"),
            "--limit",
            "20",
        ],
        stage="sft_smoke",
        layout=L,
        extra_env=af.smoke_env(L, "sft"),
    )
    S.set(sft_smoke_ok=GIT_SHA)
print("SFT smoke:", S.get("sft_smoke_ok") or "skipped")
print(af.gpu_status())  # should be back near 0 MiB used: the child released the GPU

In [ ]:
# Stage 2c: full SFT (3-4 h on T4). Re-running after a disconnect resumes from the
# newest intact checkpoint on Drive. Registry is snapshotted to Drive every 10 min.
if S.get("sft_run") and not FORCE["sft"]:
    print("SFT already complete:", S.get("sft_run"))
else:
    cmd = list(SFT_CMD)
    prev = af.latest_resumable(L.artifacts_dir, "sft")
    if prev is not None:
        print(f"Resuming {prev.run_id} from {prev.latest_checkpoint.name}")
        cmd += ["--resume", str(prev.latest_checkpoint)]  # absolute path: see Part 14 D5
        af.mark_failed(prev.run_id, "colab disconnect; resumed in a new run", layout=L)
    out = af.run(cmd, stage="sft", layout=L, heartbeat=lambda: af.persist(L))
    done = af.latest_complete(L.artifacts_dir, "sft")
    run_id = af.parse_run_id(out) or (done.run_id if done else None)
    if not run_id:
        raise RuntimeError("SFT finished but no run id found; set it with S.set(sft_run='...').")
    S.set(sft_run=run_id)
    print("SFT run:", run_id)
print(af.persist(L))

In [ ]:
# Stage 2d: did it learn? Peak VRAM and the last probe generation per prompt.
run_id = S.require("sft_run")
logs = REPO / "logs"
vram = af.log_records(logs, event="vram_step", run_id=run_id)
if vram:
    peak = max(vram, key=lambda r: r.get("peak_gb", 0))
    print(f"Peak VRAM: {peak['peak_gb']} GB of {peak.get('total_gb')} GB (step {peak.get('step')})")
last: dict[str, dict] = {}
for rec in af.log_records(logs, event="probe_result", run_id=run_id):
    last[rec["prompt"]] = rec
for prompt, rec in last.items():
    print(f"\n[step {rec.get('step')}] Q: {prompt}\nA: {rec.get('response', '')[:400]}")
if not vram and not last:
    print("No structured records for this run in logs/. Check logging.format is json.")

In [ ]:
# Stage 3a: DPO command, adapted to the flags your CLI actually exposes.
DPO_CMD = [
    "alignforge",
    "train",
    "dpo",
    "--config",
    DPO_CFG,
    "--model-config",
    MODEL_CFG,
    "--sft-run",
    S.require("sft_run"),
    "--pref-hash",
    S.require("pref_hash"),
]
DPO_FLAGS = {
    f: af.has_flag(["alignforge", "train", "dpo"], f, layout=L)
    for f in ("--dry-run", "--limit", "--resume")
}
print("train dpo supports:", DPO_FLAGS)
if DPO_FLAGS["--dry-run"]:
    af.run([*DPO_CMD, "--dry-run"], stage="dpo_dry", layout=L)

In [ ]:
# Stage 3b: DPO smoke test: proves policy + adapter-disable reference fit in VRAM.
if RUN_SMOKE_TESTS and DPO_FLAGS["--limit"] and not S.get("dpo_smoke_ok"):
    af.run(
        [*DPO_CMD, "--limit", "32"], stage="dpo_smoke", layout=L, extra_env=af.smoke_env(L, "dpo")
    )
    S.set(dpo_smoke_ok=GIT_SHA)
print("DPO smoke:", S.get("dpo_smoke_ok") or "skipped (no --limit flag or disabled)")

In [ ]:
# Stage 3c: full DPO, with the same resume + heartbeat mechanics as SFT.
if S.get("dpo_run") and not FORCE["dpo"]:
    print("DPO already complete:", S.get("dpo_run"))
else:
    cmd = list(DPO_CMD)
    prev = af.latest_resumable(L.artifacts_dir, "dpo")
    if prev is not None and DPO_FLAGS["--resume"]:
        print(f"Resuming {prev.run_id} from {prev.latest_checkpoint.name}")
        cmd += ["--resume", str(prev.latest_checkpoint)]
        af.mark_failed(prev.run_id, "colab disconnect; resumed in a new run", layout=L)
    out = af.run(cmd, stage="dpo", layout=L, heartbeat=lambda: af.persist(L))
    done = af.latest_complete(L.artifacts_dir, "dpo")
    run_id = af.parse_run_id(out) or (done.run_id if done else None)
    if not run_id:
        raise RuntimeError("DPO finished but no run id found; set it with S.set(dpo_run='...').")
    S.set(dpo_run=run_id)
    print("DPO run:", run_id)
print(af.persist(L))

In [ ]:
# Stage 3d: implicit KL trajectory. Should rise then plateau; still climbing = drift.
kl = af.log_records(REPO / "logs", key="implicit_kl", run_id=S.require("dpo_run"))
if kl:
    picks = kl[:: max(1, len(kl) // 8)]
    if picks[-1] is not kl[-1]:
        picks.append(kl[-1])
    for r in picks:
        print(f"step {r.get('step', '?'):>6}  implicit_kl={r['implicit_kl']:.3f}")
else:
    print("No implicit_kl records found for this run.")

In [ ]:
# Stage 3e (optional): beta sweep on 500 pairs. Roughly 3x a short DPO run.
if RUN_BETA_SWEEP and not S.get("beta_sweep_done"):
    af.run(
        [
            "alignforge",
            "train",
            "dpo-sweep",
            "--sft-run",
            S.require("sft_run"),
            "--pref-hash",
            S.require("pref_hash"),
            "--betas",
            "0.05,0.1,0.3",
            "--limit",
            "500",
        ],
        stage="dpo_sweep",
        layout=L,
        heartbeat=lambda: af.persist(L),
    )
    S.set(beta_sweep_done=True)
    print(af.persist(L))
else:
    print("beta sweep:", "done" if S.get("beta_sweep_done") else "skipped")

In [ ]:
# Stage 4a: register base / SFT / DPO as servable models (transformers backend).
sft_run, dpo_run = S.require("sft_run"), S.require("dpo_run")
if S.get("published") == [sft_run, dpo_run] and not FORCE["publish"]:
    print("Already published for these runs.")
else:
    for args in (
        [
            "--run",
            sft_run,
            "--as",
            "base",
            "--display",
            "Base (Qwen2.5-1.5B)",
            "--backend",
            "transformers",
            "--weights",
            BASE_WEIGHTS,
            "--sort",
            "1",
        ],
        [
            "--run",
            sft_run,
            "--as",
            "sft",
            "--display",
            "SFT",
            "--backend",
            "transformers",
            "--sort",
            "2",
        ],
        [
            "--run",
            dpo_run,
            "--as",
            "dpo",
            "--display",
            "DPO (beta=0.1)",
            "--backend",
            "transformers",
            "--sort",
            "3",
        ],
    ):
        af.run(["alignforge", "registry", "publish", *args], stage=f"publish_{args[3]}", layout=L)
    S.set(published=[sft_run, dpo_run])
af.run(["alignforge", "registry", "list"], stage="registry_list", layout=L)
print(af.persist(L))

In [ ]:
# Stage 4b: generate eval responses for all three models (~45 min on T4).
published = S.require("published")
if S.get("generated") == published and not FORCE["generate"]:
    print("Generations already exist for these runs.")
else:
    af.run(
        ["alignforge", "eval", "generate", "--models", "base,sft,dpo"],
        stage="eval_generate",
        layout=L,
        heartbeat=lambda: af.persist(L),
    )
    S.set(generated=published)
print(af.persist(L))

In [ ]:
# Stage 4c: judge + report. Nothing downstream depends on this stage.
import shutil

if JUDGE_MODE == "skip":
    print("Judging skipped (JUDGE_MODE='skip'). Set it to 'real' later and re-run this cell.")
elif JUDGE_MODE == "echo":
    # Fake verdicts into a throwaway dir, never the real one: if judging ever resumes
    # by skipping existing files, echo output there would be mistaken for real verdicts.
    scratch = L.local_root / "echo_judgements"
    shutil.rmtree(scratch, ignore_errors=True)
    af.run(
        [
            "alignforge",
            "eval",
            "judge",
            "--models",
            "base,sft,dpo",
            "--backend",
            "echo",
            "--judgements-dir",
            str(scratch),
            "--limit",
            "5",
        ],
        stage="eval_judge_echo",
        layout=L,
    )
    print(f"Echo judging OK: fake verdicts in {scratch}, not recorded in state.json.")
elif not os.environ.get("ALIGNFORGE_JUDGE_API_KEY"):
    print(
        "JUDGE_MODE='real' but ALIGNFORGE_JUDGE_API_KEY is missing: add the secret, "
        "re-run the secrets cell, then this one."
    )
elif S.get("judged") == S.require("generated") and not FORCE["judge"]:
    print("Already judged:", S.get("report"))
else:
    af.run(
        [
            "alignforge",
            "eval",
            "judge",
            "--models",
            "base,sft,dpo",
            "--backend",
            "openai",
            "--judge-model",
            JUDGE_MODEL,
            "--judge-base-url",
            JUDGE_BASE_URL,
        ],
        stage="eval_judge",
        layout=L,
    )
    af.run(
        ["alignforge", "eval", "report", "--models", "base,sft,dpo"], stage="eval_report", layout=L
    )
    reports = sorted((REPO / "reports").glob("eval_*.md"), key=lambda p: p.stat().st_mtime)
    S.set(judged=S.get("generated"), report=str(reports[-1].relative_to(REPO)) if reports else None)
    print("Report:", S.get("report"))
print(af.persist(L))

In [ ]:
# Stage 4c: judge + report. Needs no GPU: a CPU runtime with REQUIRE_GPU=False works too.
"""
if not os.environ.get("ALIGNFORGE_JUDGE_API_KEY"):
    print("ALIGNFORGE_JUDGE_API_KEY missing: add the secret, re-run the secrets cell and this one.")
elif S.get("judged") == S.require("generated") and not FORCE["judge"]:
    print("Already judged:", S.get("report"))
else:
    af.run(["alignforge", "eval", "judge", "--models", "base,sft,dpo", "--backend", "openai",
            "--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL],
           stage="eval_judge", layout=L
    )
    af.run(
        ["alignforge", "eval", "report", "--models", "base,sft,dpo"], stage="eval_report", layout=L
    )
    reports = sorted((REPO / "reports").glob("eval_*.md"), key=lambda p: p.stat().st_mtime)
    S.set(judged=S.get("generated"), report=str(reports[-1].relative_to(REPO)) if reports else None)
    print("Report:", S.get("report"))
print(af.persist(L))
"""

In [ ]:
# Stage 5: merge -> GGUF -> Q4_K_M. llama.cpp is rebuilt per VM (~4 min), outputs land on Drive.
if S.get("gguf") and not FORCE["export"]:
    print("GGUF already exported:", S.get("gguf"))
else:
    af.run(["bash", "scripts/setup_llama_cpp.sh"], stage="llama_cpp", layout=L)
    af.run(
        ["alignforge", "export", "gguf", "--dpo-run", S.require("dpo_run"), "--config", EXPORT_CFG],
        stage="export_gguf",
        layout=L,
    )
    ggufs = sorted(
        [*L.models_dir.rglob("*.gguf"), *L.artifacts_dir.rglob("*.gguf")],
        key=lambda p: p.stat().st_mtime,
    )
    for p in ggufs:
        print(f"{p.stat().st_size / 1e9:6.2f} GB  {p}")
    quant = [p for p in ggufs if "q4" in p.name.lower()]
    S.set(gguf=str((quant or ggufs)[-1]) if ggufs else None)
print(af.persist(L))

In [ ]:
# Stage 6 (optional): adapter + GGUF + model card to the HuggingFace Hub. Needs a write HF_TOKEN.
if RUN_HF_UPLOAD:
    af.run(
        [
            sys.executable,
            "scripts/upload_to_hub.py",
            "--dpo-run",
            S.require("dpo_run"),
            "--repo-id",
            HF_REPO_ID,
        ],
        stage="hf_upload",
        layout=L,
    )
else:
    print("HF upload skipped (RUN_HF_UPLOAD=False).")

In [ ]:
# Stage 7: ship results through the PR cycle: reports/ + evals/ on a fresh branch.
import time

if not PUSH_RESULTS:
    print(f"Results are on Drive at {L.mirror_dir}/reports (download and commit locally).")
elif S.get("pushed_for") == S.get("judged"):
    print("Results for these runs were already pushed:", S.get("results_branch"))
else:
    if not _tok:
        raise RuntimeError("PUSH_RESULTS needs a GH_TOKEN secret with Contents: write.")
    branch = f"exp/colab-results-{time.strftime('%Y%m%d-%H%M%S')}"
    git("checkout", "-b", branch)
    git("add", "reports", "evals")
    if git("diff", "--cached", "--name-only"):
        git(
            "-c",
            f"user.name={GIT_NAME}",
            "-c",
            f"user.email={GIT_EMAIL}",
            "commit",
            "-m",
            f"exp(eval): results for sft={S.get('sft_run')} dpo={S.get('dpo_run')} @ {GIT_SHA}",
        )
        git("push", "--quiet", "origin", branch)
        S.set(pushed_for=S.get("judged"), results_branch=branch)
        print(
            f"Open the PR: https://github.com/{GITHUB_USER}/{REPO_NAME}/compare/{branch}?expand=1"
        )
    else:
        print("Nothing new in reports/ or evals/.")
    # Stay on this branch: checking out GIT_REF would rewrite reports/ with older
    # files carrying newer mtimes, and the next persist() would copy them over Drive.

In [ ]:
# End of session: final snapshot and a summary to paste into your notes.
print(af.persist(L))
af.status(L, S)